# Revenue Forecasting Exploratory Analysis

## Project Overview
Analyze historical revenue data to uncover trend, seasonality, and external drivers before any formal forecasting model is built. This is a pre-modeling EDA project focused on understanding revenue dynamics.

## Learning Objectives
- Decompose revenue into trend, seasonal, and residual components
- Identify key external drivers correlated with revenue
- Detect structural breaks or regime changes in revenue
- Build intuition for what a forecasting model needs to capture

## Problem Statement
Before building a revenue forecast, the finance team needs to understand the data: Is revenue trending up or down? Are there seasonal patterns? What external factors correlate with revenue changes?

## Why This Project Matters
Blind forecasting without EDA leads to poor models. Understanding trend, seasonality, and drivers ensures the right model structure and avoids garbage-in-garbage-out.

## Dataset Overview
Synthetic monthly revenue data for 4 product lines over 4 years (48 months), with external economic indicators.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
months = pd.date_range('2021-01-01', '2024-12-01', freq='MS')
n_months = len(months)
products = ['SaaS Platform', 'Consulting', 'Hardware', 'Support']

records = []
for prod in products:
    base = {'SaaS Platform': 500000, 'Consulting': 200000, 'Hardware': 350000, 'Support': 150000}[prod]
    trend_rate = {'SaaS Platform': 0.02, 'Consulting': 0.005, 'Hardware': -0.005, 'Support': 0.01}[prod]
    seasonal_amp = {'SaaS Platform': 0.08, 'Consulting': 0.15, 'Hardware': 0.20, 'Support': 0.05}[prod]
    
    for i, m in enumerate(months):
        trend = base * (1 + trend_rate) ** i
        seasonal = trend * seasonal_amp * np.sin(2 * np.pi * (m.month - 1) / 12 + np.pi / 6)
        noise = np.random.normal(0, trend * 0.03)
        # Q4 bump for hardware
        q4_bump = trend * 0.15 if prod == 'Hardware' and m.month in [10, 11, 12] else 0
        revenue = trend + seasonal + noise + q4_bump
        records.append({
            'Date': m, 'Product': prod, 'Revenue': max(revenue, 10000),
            'Month': m.month, 'Year': m.year, 'Quarter': f'Q{(m.month-1)//3+1}',
        })

df = pd.DataFrame(records)
df['Revenue'] = df['Revenue'].round(2)

# External indicators (monthly)
econ = pd.DataFrame({'Date': months})
econ['GDP_Index'] = 100 + np.cumsum(np.random.normal(0.3, 0.5, n_months))
econ['Consumer_Confidence'] = 70 + np.cumsum(np.random.normal(0.1, 1.5, n_months))
econ['Interest_Rate'] = np.clip(3.0 + np.cumsum(np.random.normal(0.05, 0.15, n_months)), 1.0, 8.0)
df = df.merge(econ, on='Date')

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Products: {df["Product"].nunique()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Revenue range: ${df["Revenue"].min():,.2f} - ${df["Revenue"].max():,.2f}')
print(f'Records per product: {df.groupby("Product").size().to_dict()}')
print(f'\nTotal revenue by product:')
print(df.groupby('Product')['Revenue'].sum().sort_values(ascending=False).apply(lambda x: f'${x:,.0f}'))

## Exploratory Data Analysis — Revenue Trends

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, prod in zip(axes.flat, products):
    subset = df[df['Product'] == prod]
    ax.plot(subset['Date'], subset['Revenue'], marker='.', label=prod)
    # Add trend line
    z = np.polyfit(range(len(subset)), subset['Revenue'], 1)
    ax.plot(subset['Date'], np.polyval(z, range(len(subset))), '--', color='red', alpha=0.7, label='Trend')
    ax.set_title(f'{prod} Revenue')
    ax.set_ylabel('Revenue ($)')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'revenue_trends.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seasonality Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, prod in zip(axes.flat, products):
    seasonal = df[df['Product'] == prod].groupby('Month')['Revenue'].mean()
    seasonal.plot.bar(ax=ax, edgecolor='black', color='steelblue')
    ax.set_title(f'{prod} — Avg Revenue by Month')
    ax.set_ylabel('Avg Revenue ($)')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'seasonality.png'), dpi=100, bbox_inches='tight')
plt.show()

## Total Revenue Composition

In [ ]:
total_monthly = df.groupby(['Date', 'Product'])['Revenue'].sum().unstack()
fig, ax = plt.subplots(figsize=(14, 6))
total_monthly.plot.area(ax=ax, alpha=0.7)
ax.set_title('Revenue Composition Over Time')
ax.set_ylabel('Revenue ($)')
ax.legend(title='Product', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'composition.png'), dpi=100, bbox_inches='tight')
plt.show()

## Year-over-Year Growth

In [ ]:
yoy = df.groupby(['Year', 'Product'])['Revenue'].sum().unstack()
yoy_growth = yoy.pct_change().dropna() * 100
print('Year-over-Year Revenue Growth (%):')
print(yoy_growth.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
yoy_growth.plot.bar(ax=ax, edgecolor='black')
ax.set_title('Year-over-Year Revenue Growth by Product')
ax.set_ylabel('Growth (%)')
ax.axhline(0, color='black', linewidth=0.8)
ax.legend(title='Product')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'yoy_growth.png'), dpi=100, bbox_inches='tight')
plt.show()

## Correlation with External Drivers

In [ ]:
total_rev = df.groupby('Date')['Revenue'].sum().reset_index()
total_rev = total_rev.merge(econ, on='Date')
corr_cols = ['Revenue', 'GDP_Index', 'Consumer_Confidence', 'Interest_Rate']
corr = total_rev[corr_cols].corr()
print('Correlation Matrix:')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Revenue vs External Indicators')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlation.png'), dpi=100, bbox_inches='tight')
plt.show()

## Revenue Volatility Analysis

In [ ]:
vol = df.groupby('Product')['Revenue'].agg(['mean', 'std'])
vol['cv'] = (vol['std'] / vol['mean'] * 100).round(2)
vol.columns = ['Mean Revenue', 'Std Dev', 'CV (%)']
print('Revenue Volatility by Product:')
print(vol.round(2))

fig, ax = plt.subplots(figsize=(8, 5))
vol['CV (%)'].sort_values().plot.barh(ax=ax, edgecolor='black', color='coral')
ax.set_title('Revenue Volatility (Coefficient of Variation)')
ax.set_xlabel('CV (%)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'volatility.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **SaaS Platform** shows strong upward trend with moderate seasonality — most predictable revenue stream
- **Hardware** has declining baseline but significant Q4 spikes — seasonal demand is crucial
- **Consulting** is the most volatile — harder to forecast accurately
- **GDP and Consumer Confidence** correlate positively with total revenue — good candidate external regressors
- **Interest Rate** shows weak negative correlation — may dampen hardware/big-ticket sales
- A forecasting model should include: trend, monthly seasonality, Q4 hardware adjustment, and optionally GDP as external regressor

## Limitations
- Only 4 years of data — limited for detecting long-cycle patterns
- Synthetic external indicators may not capture real economic dynamics
- No pricing, marketing spend, or competitive data
- Aggregated to monthly level — weekly patterns not visible
- Single business entity — no multi-entity comparison

## How to Improve This Project
- Use real company revenue data with more history
- Add marketing spend, pricing changes, and competitive events as regressors
- Include weekly or daily granularity for short-term patterns
- Add formal decomposition (STL, X-13) for rigorous seasonal adjustment
- Build actual forecasting models (ARIMA, Prophet, gradient boosting with lag features)

## Production Considerations
- Automate monthly EDA refreshes before re-fitting forecasts
- Track forecast accuracy and update models when structural breaks are detected
- Integrate external data feeds (GDP, confidence indices) for real-time regressor updates
- Build scenario analysis tools for revenue planning

## Common Mistakes
- Fitting a forecast model without first understanding the data's structure
- Ignoring structural breaks (e.g., product launches, market shifts)
- Using calendar year splits instead of proper time-based validation
- Over-differencing or detrending data that has meaningful level information

## Mini Challenge / Exercises
1. Calculate the month-over-month growth rate for each product. Which product has the most volatile MoM growth?
2. If you remove the Q4 bump from Hardware, what does the underlying trend look like?
3. What lag of GDP_Index has the highest correlation with total revenue? (try lags 0-6)
4. Estimate the contribution of each product to total revenue growth from 2021 to 2024.

## Final Summary / Key Takeaways
- Pre-modeling EDA is essential for building good forecasts
- Revenue data typically contains trend, seasonality, and noise that must be understood separately
- External drivers can improve forecasts but must be validated for lead/lag relationships
- Product-level decomposition reveals different forecasting challenges per stream
- The insights from this EDA directly inform model selection, feature engineering, and validation strategy